# Buffer Pool Hit Rates and What a Scan Does To Them

Companion notebook for the [Buffer Pool and Page Replacement](https://codemia.io/courses/database_internals/dbi_buffer_pool_and_page_replacement) lesson on Codemia.

**What you will build**: four replacement policies, three workloads, and the hit rate each pairing
produces. By the end you will be able to say how much memory a given hit rate costs, why a single
large scan can halve the hit rate of a workload that has nothing to do with it, and what the
engines that resist that are actually doing differently.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import heapq
from collections import OrderedDict, deque

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)

PAGES = 100000     # pages in the table, far more than any pool size here
ACCESSES = 120000  # page requests per run

## Concept recap

A buffer pool is a fixed number of page sized frames and a rule for deciding which page to throw
out when a new one is needed. Every engine has one, every engine's rule is a variation on the same
few ideas, and the difference between them shows up only under particular workloads.

Least recently used is the obvious rule and is close to optimal when the workload has a stable hot
set. It has one well known failure: a sequential scan touches a large number of pages exactly once
each, every one of them is the most recently used page for a moment, and the hot set is pushed out
by data that will never be read again. The scan runs no faster for having done this, and everything
else runs slower afterwards.

Clock approximates least recently used with one reference bit per frame and a rotating hand, which
is what real engines implement because it needs no list surgery on a hit. It inherits the same
weakness. The policies that survive a scan all do the same thing in different clothing: they refuse
to promote a page on its first touch, so a page has to be referenced twice before it can displace
anything that earned its place.

## Four policies

In [ ]:
class LRU:
    """Textbook least recently used, in an ordered dictionary."""
    name = "LRU"

    def __init__(self, size):
        self.size = size
        self.pool = OrderedDict()

    def access(self, page):
        if page in self.pool:
            self.pool.move_to_end(page)
            return True
        if len(self.pool) >= self.size:
            self.pool.popitem(last=False)
        self.pool[page] = True
        return False


class Clock:
    """What engines actually implement: one reference bit per frame and a rotating hand."""
    name = "CLOCK"

    def __init__(self, size):
        self.size = size
        self.frames = [None] * size
        self.ref = [False] * size
        self.where = {}
        self.hand = 0

    def access(self, page):
        slot = self.where.get(page)
        if slot is not None:
            self.ref[slot] = True
            return True
        while True:
            victim = self.frames[self.hand]
            if victim is None or not self.ref[self.hand]:
                break
            self.ref[self.hand] = False
            self.hand = (self.hand + 1) % self.size
        if victim is not None:
            del self.where[victim]
        self.frames[self.hand] = page
        self.ref[self.hand] = False
        self.where[page] = self.hand
        self.hand = (self.hand + 1) % self.size
        return False


class LRU2:
    """Evict by the second most recent reference, so one touch is not enough to stay.

    The heap holds stale entries on purpose and they are discarded when they surface,
    which keeps an eviction logarithmic rather than a scan of the whole pool.
    """
    name = "LRU-2"
    FAR = -10 ** 15

    def __init__(self, size):
        self.size = size
        self.pool = {}      # page -> (last, second_last)
        self.heap = []
        self.t = 0

    def _key(self, last, second):
        return second if second is not None else self.FAR + last

    def access(self, page):
        self.t += 1
        rec = self.pool.get(page)
        hit = rec is not None
        if hit:
            last, _ = rec
            self.pool[page] = (self.t, last)
        else:
            if len(self.pool) >= self.size:
                self._evict()
            self.pool[page] = (self.t, None)
        last, second = self.pool[page]
        heapq.heappush(self.heap, (self._key(last, second), page))
        return hit

    def _evict(self):
        while self.heap:
            key, page = heapq.heappop(self.heap)
            rec = self.pool.get(page)
            if rec is None:
                continue
            if key == self._key(rec[0], rec[1]):
                del self.pool[page]
                return
        self.pool.pop(next(iter(self.pool)))


class TwoQueue:
    """A probationary queue in front of the main one, which is how a scan is kept out.

    A page seen once sits in the small first in first out queue. Only a second reference
    promotes it into the main least recently used queue, and a scan never gets one.
    """
    name = "2Q"

    def __init__(self, size, probation=0.25):
        self.a1 = deque()
        self.a1set = set()
        self.a1cap = max(1, int(size * probation))
        self.main = OrderedDict()
        self.maincap = size - self.a1cap

    def access(self, page):
        if page in self.main:
            self.main.move_to_end(page)
            return True
        if page in self.a1set:
            self.a1set.discard(page)
            self.a1.remove(page)
            if len(self.main) >= self.maincap:
                self.main.popitem(last=False)
            self.main[page] = True
            return True
        if len(self.a1) >= self.a1cap:
            self.a1set.discard(self.a1.popleft())
        self.a1.append(page)
        self.a1set.add(page)
        return False


POLICIES = [LRU, Clock, LRU2, TwoQueue]

In [ ]:
def hit_rate(policy_cls, trace, size):
    pool = policy_cls(size)
    hits = 0
    for page in trace:
        hits += pool.access(int(page))
    return hits / len(trace)

## Three workloads

In [ ]:
def skewed(n=ACCESSES, pages=PAGES, a=1.2):
    """A hot set with a long tail, which is what most transactional tables look like."""
    ranks = np.minimum(rng.zipf(a, n), pages) - 1
    identity = rng.permutation(pages)      # the hot pages are not pages 0, 1, 2
    return identity[ranks]


def cyclic(n=ACCESSES, span=6000):
    """The same pages in the same order forever, a span larger than the pool."""
    return np.tile(np.arange(span), n // span + 1)[:n]


def skewed_with_scan(n=ACCESSES, hot=2000, scan_every=10):
    """A transactional hot set, interrupted by one page of a long scan in ten."""
    hot_pages = rng.integers(0, hot, size=n)
    scan_pages = hot + np.arange(n)
    pick_scan = (np.arange(n) % scan_every) == 0
    return np.where(pick_scan, scan_pages, hot_pages)


traces = {
    "skewed": skewed(),
    "cyclic scan": cyclic(),
    "skewed plus scan": skewed_with_scan(),
}

for name, tr in traces.items():
    print("%-18s %7d accesses, %6d distinct pages" % (name, len(tr), len(np.unique(tr))))

## Hit rate against pool size

In [ ]:
sizes = [250, 500, 1000, 2000, 4000, 8000]
curves = {}
for cls in POLICIES:
    curves[cls.name] = [hit_rate(cls, traces["skewed"], s) for s in sizes]

print("%-8s" % "pool" + "".join("%10s" % c.name for c in POLICIES))
for i, s in enumerate(sizes):
    print("%-8d" % s + "".join("%9.1f%%" % (100 * curves[c.name][i]) for c in POLICIES))

In [ ]:
plt.figure(figsize=(8, 4))
for cls in POLICIES:
    plt.plot(sizes, [100 * v for v in curves[cls.name]], "o-", lw=2, label=cls.name)
plt.xscale("log", base=2)
plt.xlabel("frames in the pool")
plt.ylabel("hit rate, percent")
plt.title("a skewed workload: every policy looks much the same")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

On a workload with a stable hot set the four policies are within a couple of points of each other,
which is the first useful result: if this is your workload, the replacement policy is not where your
hit rate is. The pool size is.

Notice the shape of the curve rather than its height. Doubling the pool buys a lot at the left hand
end and very little at the right, because the hot set fits long before the pool is large enough to
hold the tail. The knee in that curve is the memory figure worth arguing for in a budget, and
everything past it is being spent on pages that are read once.

## The same policies, with a scan running

In [ ]:
print("%-18s" % "workload" + "".join("%10s" % c.name for c in POLICIES))
for name, tr in traces.items():
    row = [hit_rate(cls, tr, 2000) for cls in POLICIES]
    print("%-18s" % name + "".join("%9.1f%%" % (100 * v) for v in row))
print()
print("pool of 2000 frames in every column")

Two failures in one table, and the second is not the one usually advertised.

The cyclic scan takes every policy to zero. A loop over six thousand pages through a pool of two
thousand evicts each page shortly before it is needed again, and no amount of recency bookkeeping
helps, because every page has exactly the same access pattern as every other page. This is worth
seeing rather than assuming: recency cannot rank pages that are indistinguishable.

The skewed workload with one scanned page in ten is the case that shows up in production. Least
recently used loses twelve points to it, and only the policy that ranks by the second most recent
reference holds its hit rate, because the scanned pages never acquire one. The two queue policy
barely helps here, and the reason is visible in its own parameters: a quarter of the pool is
reserved for probation, which leaves fifteen hundred frames for a hot set of two thousand pages.
It defended the hot set from the scan and then could not hold the hot set. The sweep below is
that trade in one table.

## The one case where evicting the newest page is correct

In [ ]:
class MRU:
    """Evict the most recently used page, which is almost always wrong."""
    name = "MRU"

    def __init__(self, size):
        self.size = size
        self.pool = OrderedDict()

    def access(self, page):
        if page in self.pool:
            self.pool.move_to_end(page)
            return True
        if len(self.pool) >= self.size:
            self.pool.popitem(last=True)
        self.pool[page] = True
        return False


print("%-18s %10s %10s" % ("workload", "LRU", "MRU"))
for name in ("skewed", "cyclic scan"):
    print("%-18s %9.1f%% %9.1f%%" % (name,
                                     100 * hit_rate(LRU, traces[name], 2000),
                                     100 * hit_rate(MRU, traces[name], 2000)))
print()
print("the pool holds 2000 of the 6000 pages in the loop, and MRU keeps the same 2000")

Evicting the page you just touched is the correct policy for a loop larger than the pool, because
the page you just touched is the one you will need last. It turns a hit rate of zero into roughly
the fraction of the loop that fits, and it is catastrophic on anything else, which is why no engine
makes it the default and several make it available to a plan that knows it is scanning.

That is the general shape of the answer. Replacement policies cannot infer intent from an access
stream that gives them nothing to work with. The engines that handle scans well handle them by being
told, either by the planner that chose the scan or by a page that has only ever been touched once.

## Variations: how much of the pool a scan is allowed to take

In [ ]:
print("%-12s %10s %10s" % ("probation", "skewed", "skewed plus scan"))
for frac in (0.05, 0.15, 0.25, 0.5, 0.75):
    a = hit_rate(lambda s, f=frac: TwoQueue(s, probation=f), traces["skewed"], 2000)
    b = hit_rate(lambda s, f=frac: TwoQueue(s, probation=f), traces["skewed plus scan"], 2000)
    print("%-12.2f %9.1f%% %9.1f%%" % (frac, 100 * a, 100 * b))

The probationary fraction is the one setting this family of policies really has, and it is a bet
about how much of the pool should be reserved for pages that have proved nothing yet.

Too small and a page that is genuinely becoming hot is evicted before it can be referenced a second
time, which is the first row. Too large and the main area can no longer hold the hot set, which is
every row after the middle one. The optimum here is nearer fifteen percent than the twenty five that
implementations usually ship, for a specific reason worth noticing: this hot set is two thousand
pages and this pool is two thousand frames, so every frame given to probation is taken from a hot
set that already does not fit. Make the pool larger and the curve flattens.

The real lesson is the one the engines took: the scan does not need a special case, a special hint
or a configuration flag. Requiring two references before promotion handles it as a side effect.

## What to try next

- Add dirty pages and a write back cost, so that evicting a modified page means a write as well as a
  read. The ranking of the policies changes when eviction is not free.
- Give the scan its own small ring of frames, which is what a real engine does for a sequential scan
  it can recognise, and compare that against the two touch policies.
- Replay a trace with a hot set that moves, a quarter of the keys changing every ten thousand
  accesses, and see which policy adapts and which keeps defending pages that went cold.
- Work out, from the knee in the first curve, what the hit rate would be if the pool held the whole
  hot set but nothing else, and compare it against what doubling the memory actually bought.